# 17 · Scenario Runner 与 Log Replay：从一条轨迹到可回归的测试集

只在静态日志上计算 ADE/FDE 或控制误差，无法观察 planner 偏离专家轨迹之后的误差累积。L4 开发需要把道路事件变成可重复执行的 scenario，并区分：

- `log_replay`：ego 和其他交通参与者都按记录回放；
- `closed_loop_nonreactive`：ego 由当前策略控制，其他参与者按记录运行；
- `closed_loop_reactive`：ego 和其他参与者都根据当前状态作出反应。

这里用一个轻量 runner 模拟前车急刹场景，接口可以进一步接到 nuPlan、NAVSIM 或 CARLA ScenarioRunner。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, replace

plt.rcParams["figure.figsize"] = (10, 4.5)
plt.rcParams["axes.grid"] = True

@dataclass(frozen=True)
class Scenario:
    name: str
    initial_gap_m: float = 28.0
    ego_speed_mps: float = 8.0
    lead_speed_mps: float = 6.0
    lead_brake_time_s: float = 4.0
    duration_s: float = 10.0
    dt: float = 0.1
    reactive_agents: bool = False


scenario = Scenario("lead_vehicle_hard_brake")


## Part A — 记录轨迹和 scenario contract

一个 scenario 不应只保存视频文件名，还需要初始状态、交通参与者行为、地图/天气条件、随机种子和可计算的 pass/fail criteria。下面的最小 contract 只包含纵向运动。


In [ ]:
def lead_profile(s, sc):
    speed = sc.lead_speed_mps if s < sc.lead_brake_time_s else 0.0
    return speed


def run_scenario(sc, mode="closed_loop_reactive", policy="cautious"):
    time_s = np.arange(0.0, sc.duration_s, sc.dt)
    lead_x = sc.initial_gap_m
    ego_x = 0.0
    ego_speed = sc.ego_speed_mps
    rows = []
    for now in time_s:
        lead_speed = lead_profile(now, sc)
        gap = lead_x - ego_x
        if mode == "closed_loop_reactive" and sc.reactive_agents and gap < 16.0:
            lead_speed = min(lead_speed, max(0.0, ego_speed - 1.0))

        if mode == "log_replay":
            target_speed = sc.ego_speed_mps
        elif policy == "naive":
            target_speed = sc.ego_speed_mps
        else:
            target_speed = sc.ego_speed_mps
            if gap < 18.0:
                target_speed = min(target_speed, max(0.0, lead_speed - 1.0))
            if gap < 8.0:
                target_speed = 0.0

        acceleration = np.clip((target_speed - ego_speed) * 1.4, -4.0, 2.0)
        if mode == "log_replay":
            acceleration = 0.0
        ego_speed = max(0.0, ego_speed + acceleration * sc.dt)
        ego_x += ego_speed * sc.dt
        lead_x += lead_speed * sc.dt
        rows.append({"time_s": now, "ego_x": ego_x, "lead_x": lead_x,
                     "ego_speed": ego_speed, "lead_speed": lead_speed,
                     "gap": lead_x - ego_x, "acceleration": acceleration})
    result = pd.DataFrame(rows)
    result["collision"] = result["gap"] < 2.0
    return result


def metrics(result):
    acceleration = result["acceleration"].to_numpy()
    jerk = np.diff(acceleration, prepend=acceleration[0]) / 0.1
    return {
        "collision": bool(result["collision"].any()),
        "min_gap_m": float(result["gap"].min()),
        "progress_m": float(result["ego_x"].iloc[-1]),
        "max_decel_mps2": float(-min(acceleration.min(), 0.0)),
        "max_jerk_mps3": float(np.abs(jerk).max()),
    }


results = {}
for mode in ["log_replay", "closed_loop_nonreactive", "closed_loop_reactive"]:
    results[mode] = run_scenario(replace(scenario, reactive_agents=True), mode=mode)
display(pd.DataFrame({mode: metrics(result) for mode, result in results.items()}))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for mode, result in results.items():
    axes[0].plot(result["time_s"], result["gap"], label=mode)
    axes[1].plot(result["ego_x"], result["lead_x"], label=mode)
axes[0].axhline(2.0, color="red", linestyle="--", label="collision threshold")
axes[0].set(xlabel="time / s", ylabel="lead gap / m", title="Open-loop and closed-loop gap")
axes[1].set(xlabel="ego x / m", ylabel="lead x / m", title="Trajectory interaction")
axes[0].legend(fontsize=8)
axes[1].legend(fontsize=8)
plt.show()


## Part B — Counterfactual scenario sweep

真实 data engine 会从 intervention、近碰撞、planner disagreement 等事件挖掘场景，并生成参数化变体。这里扫描初始 gap 和急刹时间，得到一个小型 regression matrix。


In [ ]:
rows = []
for gap in np.linspace(12, 36, 9):
    for brake_time in np.linspace(2.0, 6.0, 9):
        sc = replace(scenario, initial_gap_m=float(gap), lead_brake_time_s=float(brake_time), reactive_agents=True)
        result = run_scenario(sc, mode="closed_loop_reactive", policy="cautious")
        rows.append({"gap_m": gap, "brake_time_s": brake_time, **metrics(result)})
sweep = pd.DataFrame(rows)
pivot = sweep.pivot(index="gap_m", columns="brake_time_s", values="collision")
plt.imshow(pivot.to_numpy(), origin="lower", aspect="auto", cmap="RdYlGn_r")
plt.colorbar(label="collision")
plt.xticks(range(len(pivot.columns)), [f"{x:.1f}" for x in pivot.columns], rotation=45)
plt.yticks(range(len(pivot.index)), [f"{x:.0f}" for x in pivot.index])
plt.xlabel("lead brake time / s")
plt.ylabel("initial gap / m")
plt.title("Scenario regression matrix")
plt.show()
print("collision rate:", sweep["collision"].mean())


### 练习

1. 增加横穿行人和红灯两个 scenario type；
2. 给 reactive agent 加入基于 TTC 的让行策略；
3. 将 `metrics` 扩展为 collision、off-road、progress、jerk、fallback latency 和 scenario coverage；
4. 设计一个 scenario ID，使同一变体可以在 log replay、nuPlan 和 CARLA runner 中对应；
5. 解释为什么一个 open-loop 低误差模型可能在 closed-loop 中发生碰撞。


## 完成标准

交付一张 scenario regression 表和一个失败场景回放图；不能只报告平均轨迹误差。
